In [1]:
#  Copyright (c) Microsoft Corporation.
#  Licensed under the MIT License.

import fire

import qlib
import pickle
from qlib.constant import REG_CN
from qlib.config import HIGH_FREQ_CONFIG

from qlib.utils import init_instance_by_config
from qlib.data.dataset.handler import DataHandlerLP
from qlib.data.ops import Operators
from qlib.data.data import Cal
from qlib.tests.data import GetData

from highfreq_ops import get_calendar_day, DayLast, FFillNan, BFillNan, Date, Select, IsNull, Cut
from workflow import HighfreqWorkflow

In [5]:
SPEC_CONF = {"custom_ops": [DayLast, FFillNan, BFillNan, Date, Select, IsNull, Cut], "expression_cache": None}
{**HIGH_FREQ_CONFIG, **SPEC_CONF}

{'provider_uri': '~/.qlib/qlib_data/cn_data_1min',
 'dataset_cache': None,
 'expression_cache': None,
 'region': 'cn',
 'custom_ops': [highfreq_ops.DayLast,
  highfreq_ops.FFillNan,
  highfreq_ops.BFillNan,
  highfreq_ops.Date,
  highfreq_ops.Select,
  highfreq_ops.IsNull,
  highfreq_ops.Cut]}

In [7]:
HighfreqWorkflow()._init_qlib()

2025-04-06 17:46:55.367 | WARNING  | qlib.tests.data:qlib_data:195 - Data already exists: ~/.qlib/qlib_data/cn_data_1min, the data download will be skipped
	If downloading is required: `exists_skip=False` or `change target_dir`
[170927:MainThread](2025-04-06 17:46:55,370) INFO - qlib.Initialization - [config.py:420] - default_conf: client.
[170927:MainThread](2025-04-06 17:46:55,373) INFO - qlib.Initialization - [__init__.py:74] - qlib successfully initialized based on client settings.
[170927:MainThread](2025-04-06 17:46:55,375) INFO - qlib.Initialization - [__init__.py:76] - data_path={'__DEFAULT_FREQ': PosixPath('/home/absolutex/.qlib/qlib_data/cn_data_1min')}


In [8]:
SPEC_CONF = {"custom_ops": [DayLast, FFillNan, BFillNan, Date, Select, IsNull, Cut], "expression_cache": None}

MARKET = "all"

start_time = "2020-09-15 00:00:00"
end_time = "2021-01-18 16:00:00"
train_end_time = "2020-11-30 16:00:00"
test_start_time = "2020-12-01 00:00:00"

DATA_HANDLER_CONFIG0 = {
    "start_time": start_time,
    "end_time": end_time,
    "fit_start_time": start_time,
    "fit_end_time": train_end_time,
    "instruments": MARKET,
    "infer_processors": [{"class": "HighFreqNorm", "module_path": "highfreq_processor"}],
}
DATA_HANDLER_CONFIG1 = {
    "start_time": start_time,
    "end_time": end_time,
    "instruments": MARKET,
}

task = {
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "HighFreqHandler",
                "module_path": "highfreq_handler",
                "kwargs": DATA_HANDLER_CONFIG0,
            },
            "segments": {
                "train": (start_time, train_end_time),
                "test": (
                    test_start_time,
                    end_time,
                ),
            },
        },
    },
    "dataset_backtest": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "HighFreqBacktestHandler",
                "module_path": "highfreq_handler",
                "kwargs": DATA_HANDLER_CONFIG1,
            },
            "segments": {
                "train": (start_time, train_end_time),
                "test": (
                    test_start_time,
                    end_time,
                ),
            },
        },
    },
}

def _init_qlib():
    """initialize qlib"""
    # use cn_data_1min data
    QLIB_INIT_CONFIG = {**HIGH_FREQ_CONFIG, **SPEC_CONF}
    provider_uri = QLIB_INIT_CONFIG.get("provider_uri")
    GetData().qlib_data(target_dir=provider_uri, interval="1min", region=REG_CN, exists_skip=True)
    qlib.init(**QLIB_INIT_CONFIG)

def _prepare_calender_cache():
    """preload the calendar for cache"""

    # This code used the copy-on-write feature of Linux to avoid calculating the calendar multiple times in the subprocess
    # This code may accelerate, but may be not useful on Windows and Mac Os
    Cal.calendar(freq="1min")
    get_calendar_day(freq="1min")

In [10]:
_init_qlib()
_prepare_calender_cache()

2025-04-06 17:48:38.422 | WARNING  | qlib.tests.data:qlib_data:195 - Data already exists: ~/.qlib/qlib_data/cn_data_1min, the data download will be skipped
	If downloading is required: `exists_skip=False` or `change target_dir`
[170927:MainThread](2025-04-06 17:48:38,425) INFO - qlib.Initialization - [config.py:420] - default_conf: client.
[170927:MainThread](2025-04-06 17:48:38,429) INFO - qlib.Initialization - [__init__.py:74] - qlib successfully initialized based on client settings.
[170927:MainThread](2025-04-06 17:48:38,430) INFO - qlib.Initialization - [__init__.py:76] - data_path={'__DEFAULT_FREQ': PosixPath('/home/absolutex/.qlib/qlib_data/cn_data_1min')}
